# 01 · Data Audit — MatRisk AI (Project 1A)

**Purpose.** Inspect *every* supplied file before any modelling, verify what the README claims against what the data contain,
and record every data-quality decision as **PROBLEM → DECISION → REASON**.

**Rules honoured here:** raw files are read-only (SHA-256 verified at start and end); nothing is deleted or imputed in this notebook;
every number below is computed by code from the supplied files; documented constraints (README / data dictionary / physical definitions)
are encoded as explicit, *sourced* rules in `src/data/audit.py`.

In [1]:
import sys, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
ROOT = None
for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (c / "src").exists() and (c / "data").exists():
        ROOT = c; break
sys.path.insert(0, str(ROOT))
from src.data import audit as A

RAW, TAB, FIG, REP = ROOT/"data/raw", ROOT/"outputs/tables", ROOT/"outputs/figures", ROOT/"outputs/reports"
for p in (TAB, FIG, REP): p.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 80); pd.set_option("display.max_rows", 120)

def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
RAW_HASH_START = {p.name: sha(p) for p in sorted(RAW.iterdir())}
print("Project root:", ROOT.name, "| raw files hashed:", len(RAW_HASH_START))

Project root: matrisk | raw files hashed: 8


## 1 · ZIP inventory and README

In [2]:
inv = pd.DataFrame([{"file": p.name, "type": p.suffix, "size_KB": round(p.stat().st_size/1024, 1)} for p in sorted(RAW.iterdir())])
display(inv)
readme = A.read_docx_text(RAW / A.README_FILE)
print(readme)

,file,type,size_KB
0,DS1_material_properties_5500.csv,.csv,570.5
1,DS2_commodity_prices_10yr.csv,.csv,5019.2
2,DS3_infrastructure_bridges_5000.csv,.csv,742.0
3,DS4_crossdomain_features_daily.csv,.csv,1787.7
4,DS5_element_prices_monthly.csv,.csv,166.4
5,DS6_historical_failures_2000.csv,.csv,274.3
6,MatRisk_AI_Sample_Datasets.xlsx,.xlsx,35.5
7,ReadME for Dataset.docx,.docx,18.1


MatRisk AI — Project Datasets
DS-1: Material Properties (5,500 entries, 18 columns)
File: DS1_material_properties_5500.csv
Purpose: Training data for CGNN, PINN, and ensemble models
Key columns: material_id, formula, crystal_system, spacegroup_number, formation_energy_per_atom_eV, band_gap_eV, bulk_modulus_GPa, shear_modulus_GPa, poisson_ratio, density_g_cm3, melting_point_K, is_stable, category
Physics constraints to enforce:
Elastic consistency: E = 2G(1+ν) = 3K(1−2ν)
Poisson bounds: −1 < ν < 0.5
Stability: formation_energy < 0 for stable compounds
K > 0, G > 0, density > 0
DS-2: Commodity Prices (22,952 rows, 24 columns)
File: DS2_commodity_prices_10yr.csv
Purpose: Commodity prediction model training and backtesting
Coverage: 8 commodities × ~2,870 trading days (2014–2024)Commodities: Steel_HRC, Aluminium, Copper, Nickel, Lithium_ETF, Cobalt_Index, RareEarth_ETF, Iron_Ore
Pre-computed features: daily_return, return_5d, return_21d, volatility_5d/21d/63d_ann, sma_21, sma_63, bollinger

## 2 · Load every dataset (CSV ×6, workbook blocks)

In [3]:
dfs = A.load_csvs(RAW)
xl  = A.load_xlsx_blocks(RAW / A.XLSX_FILE)
shape = pd.DataFrame([{"dataset": k, "file": A.CSV_FILES[k], "rows": len(v), "cols": v.shape[1]} for k, v in dfs.items()])
display(shape)
print("README claims: DS1 5500x18 | DS2 22952x24 | DS3 5000x24 | DS4 22952x12 | DS5 5280x5 | DS6 2000x18")
claims = {"DS1": (5500, 18), "DS2": (22952, 24), "DS3": (5000, 24), "DS4": (22952, 12), "DS5": (5280, 5), "DS6": (2000, 18)}
print("Shape matches README:", {k: (len(dfs[k]), dfs[k].shape[1]) == v for k, v in claims.items()})
print("\nWorkbook blocks parsed (generic parser, no schema assumed):")
display(pd.DataFrame([{"block": k, "rows": v.shape[0], "cols": v.shape[1]} for k, v in xl.items()]))

,dataset,file,rows,cols
0,DS1,DS1_material_properties_5500.csv,5500,18
1,DS2,DS2_commodity_prices_10yr.csv,22952,24
2,DS3,DS3_infrastructure_bridges_5000.csv,5000,24
3,DS4,DS4_crossdomain_features_daily.csv,22952,12
4,DS5,DS5_element_prices_monthly.csv,5280,5
5,DS6,DS6_historical_failures_2000.csv,2000,18


README claims: DS1 5500x18 | DS2 22952x24 | DS3 5000x24 | DS4 22952x12 | DS5 5280x5 | DS6 2000x18
Shape matches README: {'DS1': True, 'DS2': True, 'DS3': True, 'DS4': True, 'DS5': True, 'DS6': True}

Workbook blocks parsed (generic parser, no schema assumed):


,block,rows,cols
0,DS1_Material_Properties::block1,50,12
1,DS1_Material_Properties::VALIDATION CHECKS,50,6
2,DS2_Commodity_Prices::COMMODITY TICKER REFERENCE,8,6
3,DS2_Commodity_Prices::STEEL HRC SAMPLE DATA (30 TRADING DAYS),30,8
4,DS3_Infrastructure_Portfolio::block1,10,16
5,DS3_Infrastructure_Portfolio::MATERIAL PROPERTIES FOR PORTFOLIO ASSETS,10,10
6,DS4_CrossDomain_Features::block1,8,11
7,DS4_CrossDomain_Features::SIGNAL INTERPRETATION GUIDE,6,5
8,DS5_Stress_Scenarios::block1,5,10
9,DS5_Stress_Scenarios::DETAILED SHOCK PARAMETERS FOR MONTE CARLO,10,9


## 3 · Schemas, missingness, uniqueness, statistics (per dataset)

In [4]:
profiles, miss_rows = {}, []
for k, df in dfs.items():
    p = A.column_profile(df); profiles[k] = p
    print("="*110); print(f"{k}  {A.CSV_FILES[k]}   rows={len(df):,} cols={df.shape[1]}")
    display(p[["column","dtype","missing","missing_pct","n_unique","constant","near_constant","high_cardinality_cat","min","median","max","examples"]])
    for _, r in p.iterrows():
        miss_rows.append({"dataset": k, "column": r.column, "missing": r.missing, "missing_pct": r.missing_pct})
pd.concat([p.assign(dataset=k) for k, p in profiles.items()]).to_csv(TAB/"schema_profiles.csv", index=False)
missing_all = pd.DataFrame(miss_rows); missing_all.to_csv(TAB/"missing_values.csv", index=False)
display(missing_all[missing_all.missing > 0])

DS1  DS1_material_properties_5500.csv   rows=5,500 cols=18


,column,dtype,missing,missing_pct,n_unique,constant,near_constant,high_cardinality_cat,min,median,max,examples
0,material_id,object,0,0.0,5479,False,False,True,NaN,NaN,NaN,mp-121959 | mp-23248 | mp-526982
1,formula,object,0,0.0,5002,False,False,True,NaN,NaN,NaN,Ta3Au3H2 | Al3Rh4W3 | Pd3Te3
2,n_elements,int64,0,0.0,4,False,False,False,1.000,2.00000,4.0000,3 | 2 | 4
3,crystal_system,object,0,0.0,7,False,False,False,NaN,NaN,NaN,hexagonal | cubic | tetragonal
4,spacegroup_number,int64,0,0.0,230,False,False,False,1.000,165.00000,230.0000,185 | 171 | 227
5,category,object,0,0.0,15,False,False,False,NaN,NaN,NaN,Chalcogenide | Alloy | Metal
6,formation_energy_per_atom_eV,float64,0,0.0,4949,False,False,False,-6.500,-1.26490,1.5000,-0.6329 | -1.2275 | 0.1265
7,energy_above_hull_eV,float64,0,0.0,1652,False,False,False,0.000,0.03160,0.6327,0.0175 | 0.2124 | 0.0993
8,band_gap_eV,float64,0,0.0,2847,False,False,False,0.000,1.16900,12.0000,1.94 | 0.0 | 1.83
9,is_metal,int64,0,0.0,2,False,False,False,0.000,0.00000,1.0000,0 | 1


DS2  DS2_commodity_prices_10yr.csv   rows=22,952 cols=24


,column,dtype,missing,missing_pct,n_unique,constant,near_constant,high_cardinality_cat,min,median,max,examples
0,date,datetime64[ns],0,0.000,2869,False,False,False,2014-01-02 00:00:00,NaN,2024-12-31 00:00:00,2014-01-02 00:00:00 | 2014-01-03 00:00:00 | 2014-01-06 00:00:00
1,commodity,object,0,0.000,8,False,False,False,NaN,NaN,NaN,Steel_HRC | Aluminium | Copper
2,open,float64,0,0.000,22739,False,False,False,2.3632,92.299850,39197.0518,598.5489 | 573.808 | 575.2184
3,high,float64,0,0.000,22740,False,False,False,2.3757,92.846900,39650.6462,602.7788 | 578.2438 | 579.4599
4,low,float64,0,0.000,22766,False,False,False,2.3521,91.727000,39081.2978,595.2943 | 572.2654 | 573.8479
5,close,float64,0,0.000,22761,False,False,False,2.3653,92.317500,39289.9051,598.4254 | 574.4014 | 576.0968
6,volume,int64,0,0.000,6374,False,False,False,69,1830.000000,49427,1109 | 796 | 2226
7,daily_return,float64,8,0.035,19704,False,False,False,-0.109976,0.000097,0.086673,-0.040973 | 0.002947 | 0.00531
8,return_5d,float64,40,0.174,21354,False,False,False,-0.250806,0.000509,0.222218,-0.029522 | 0.021312 | 0.024813
9,return_21d,float64,168,0.732,22020,False,False,False,-0.43581,0.000484,0.452755,0.07074 | 0.127821 | 0.152341


DS3  DS3_infrastructure_bridges_5000.csv   rows=5,000 cols=24


,column,dtype,missing,missing_pct,n_unique,constant,near_constant,high_cardinality_cat,min,median,max,examples
0,bridge_id,object,0,0.0,5000,False,False,True,NaN,NaN,NaN,BR-00001 | BR-00002 | BR-00003
1,bridge_type,object,0,0.0,5,False,False,False,NaN,NaN,NaN,Pedestrian | Interstate | Local Road
2,material,object,0,0.0,11,False,False,False,NaN,NaN,NaN,Reinforced Concrete | FRP Composite | Prestressed Concrete
3,year_built,int64,0,0.0,100,False,False,False,1920,1.978000e+03,2023,1997 | 1953 | 1989
4,age_years,int64,0,0.0,100,False,False,False,1,4.600000e+01,104,27 | 71 | 35
5,design_life_years,int64,0,0.0,4,False,False,False,50,7.500000e+01,100,100 | 75 | 50
6,location,object,0,0.0,15,False,False,False,NaN,NaN,NaN,Urban Midwest | Coastal California | Coastal Florida
7,corrosion_environment,object,0,0.0,3,False,False,False,NaN,NaN,NaN,Moderate | Severe | Mild
8,corrosion_rate_mm_yr,float64,0,0.0,1052,False,False,False,0.0023,3.470000e-02,0.1489,0.0324 | 0.032 | 0.0265
9,condition_rating,int64,0,0.0,10,False,False,False,0,8.000000e+00,9,9 | 7 | 8


... (3 outputs left out)

,column,dtype,missing,missing_pct,n_unique,constant,near_constant,high_cardinality_cat,min,median,max,examples
0,date,datetime64[ns],0,0.0,132,False,False,False,2014-01-01 00:00:00,NaN,2024-12-01 00:00:00,2014-01-01 00:00:00 | 2014-02-01 00:00:00 | 2014-03-01 00:00:00
1,element,object,0,0.0,40,False,False,False,NaN,NaN,NaN,Fe | Al | Cu
2,price_usd_per_kg,float64,0,0.0,3263,False,False,False,0.09,25.63500,58286.44,0.1 | 0.09 | 2.19
3,base_price,float64,0,0.0,32,False,False,False,0.1,25.00000,55000.0,0.1 | 2.2 | 8.5
4,monthly_return,float64,0,0.0,3872,False,False,False,-1.3846,0.00955,1.2593,0.2582 | -0.1884 | -0.2059


DS6  DS6_historical_failures_2000.csv   rows=2,000 cols=18


,column,dtype,missing,missing_pct,n_unique,constant,near_constant,high_cardinality_cat,min,median,max,examples
0,event_id,object,0,0.0,2000,False,False,True,NaN,NaN,NaN,FE-00001 | FE-00002 | FE-00003
1,event_year,int64,0,0.0,19,False,False,False,2005.0000,2.014000e+03,2.023000e+03,2014 | 2022 | 2018
2,structure_type,object,0,0.0,4,False,False,False,NaN,NaN,NaN,Pipeline | Building | Bridge
3,material,object,0,0.0,11,False,False,False,NaN,NaN,NaN,FRP Composite | Stainless Steel 316L | Steel (A514 HY)
4,location,object,0,0.0,15,False,False,False,NaN,NaN,NaN,Industrial Michigan | Rural South | Coastal Carolina
5,corrosion_environment,object,0,0.0,3,False,False,False,NaN,NaN,NaN,Moderate | Mild | Severe
6,age_at_event_years,int64,0,0.0,81,False,False,False,11.0000,4.500000e+01,9.900000e+01,22 | 43 | 31
7,corrosion_rate_mm_yr,float64,0,0.0,900,False,False,False,0.0016,4.630000e-02,1.478000e-01,0.0865 | 0.007 | 0.0673
8,failure_mode,object,0,0.0,8,False,False,False,NaN,NaN,NaN,Fatigue crack | Buckling | Corrosion
9,severity,object,0,0.0,4,False,False,False,NaN,NaN,NaN,Minor | Catastrophic | Major


,dataset,column,missing,missing_pct
25,DS2,daily_return,8,0.035
26,DS2,return_5d,40,0.174
27,DS2,return_21d,168,0.732
28,DS2,volatility_5d_ann,40,0.174
29,DS2,volatility_21d_ann,168,0.732
30,DS2,volatility_63d_ann,504,2.196
31,DS2,sma_21,160,0.697
32,DS2,sma_63,496,2.161
33,DS2,bollinger_upper,160,0.697
34,DS2,bollinger_lower,160,0.697


## 4 · Categorical distributions

In [5]:
for k, df in dfs.items():
    print("="*80, k)
    for c, vc in A.categorical_distribution(df).items():
        print(f"- {c}: " + ", ".join(f"{i}={v}" for i, v in vc.items()))
print()
for k, df in dfs.items():
    v = A.category_label_variants(df)
    if len(v): print(k, "label-variant check -> columns where normalisation changes #unique:", v[v.raw_unique != v.normalised_unique].column.tolist(), "| leading/trailing-space cells:", int(v.has_leading_trailing_space.sum()))

================================================================================ DS1


- n_elements: 1=678, 2=2502, 3=1602, 4=718
- crystal_system: cubic=1615, hexagonal=1107, orthorhombic=860, tetragonal=816, monoclinic=562, trigonal=378, triclinic=162
- category: Metal=1010, Oxide=773, Semiconductor=547, Ceramic=471, Intermetallic=451, Alloy=328, Nitride=304, Halide=267, Chalcogenide=249, Sulfide=239, Perovskite=215, Carbide=214, Silicate=168, Phosphide=157, Boride=107
- is_metal: 0=3711, 1=1789
- nsites: 2=535, 4=1109, 6=814, 8=1142, 12=820, 16=532, 24=270, 32=278
- is_stable: 0=2022, 1=3478
================================================================================ DS2
- commodity: Steel_HRC=2869, Aluminium=2869, Copper=2869, Nickel=2869, Lithium_ETF=2869, Cobalt_Index=2869, RareEarth_ETF=2869, Iron_Ore=2869
================================================================================ DS3
- bridge_type: Highway=1739, Interstate=1522, Local Road=814, Railroad=516, Pedestrian=409
- material: Steel (A36)=1232, Reinforced Concrete=977, Prestressed Concrete=921, S

DS1 label-variant check -> columns where normalisation changes #unique: [] | leading/trailing-space cells: 0
DS2 label-variant check -> columns where normalisation changes #unique: [] | leading/trailing-space cells: 0
DS3 label-variant check -> columns where normalisation changes #unique: [] | leading/trailing-space cells: 0


DS4 label-variant check -> columns where normalisation changes #unique: [] | leading/trailing-space cells: 0


DS5 label-variant check -> columns where normalisation changes #unique: [] | leading/trailing-space cells: 0
DS6 label-variant check -> columns where normalisation changes #unique: [] | leading/trailing-space cells: 0


## 5 · Duplicates and candidate primary keys

In [6]:
rows = []
for k, df in dfs.items():
    rows.append({"dataset": k, "full_row_duplicates": int(df.duplicated().sum()), "candidate_keys": A.candidate_keys(df)})
keys_df = pd.DataFrame(rows); display(keys_df)

,dataset,full_row_duplicates,candidate_keys
0,DS1,0,"[(material_id, formula), (material_id, spacegroup_number)]"
1,DS2,0,"[(date, commodity)]"
2,DS3,0,"[(bridge_id,)]"
3,DS4,0,"[(date, commodity)]"
4,DS5,0,"[(date, element)]"
5,DS6,0,"[(event_id,)]"


In [7]:
d1 = dfs["DS1"]
dup = d1[d1.material_id.duplicated(keep=False)].sort_values("material_id")
print(f"DS1: {dup.material_id.nunique()} material_ids appear twice ({len(dup)} rows). Do the paired rows describe the same material?")
display(dup[["material_id","formula","crystal_system","spacegroup_number","category","formation_energy_per_atom_eV","bulk_modulus_GPa"]].head(8))
same_formula = dup.groupby("material_id").formula.nunique()
print("Duplicated ids whose two rows have DIFFERENT formulas:", int((same_formula > 1).sum()), "of", len(same_formula))
g = d1[d1.formula.duplicated(keep=False)].groupby("formula").agg(n=("material_id","size"), K_mean=("bulk_modulus_GPa","mean"), K_std=("bulk_modulus_GPa","std"), crystal_systems=("crystal_system","nunique"))
print(f"DS1: {d1.formula.duplicated().sum()} rows repeat a formula ({len(g)} formulas). Median coefficient of variation of K across same-formula rows: {(g.K_std/g.K_mean).median():.2f}")

DS1: 21 material_ids appear twice (42 rows). Do the paired rows describe the same material?


,material_id,formula,crystal_system,spacegroup_number,category,formation_energy_per_atom_eV,bulk_modulus_GPa
1745,mp-107320,Na2N3,tetragonal,130,Silicate,-3.1682,96.81
2051,mp-107320,Nb2Fe4,orthorhombic,64,Metal,0.0921,222.43
644,mp-110943,SeCl4He4,tetragonal,134,Metal,0.3555,155.30
1175,mp-110943,Se,orthorhombic,26,Metal,0.0000,283.09
3810,mp-176725,TeRuY4,tetragonal,95,Carbide,-2.3594,148.52
3271,mp-176725,Mg3Al3,cubic,214,Intermetallic,-1.7385,136.81
414,mp-194651,Ca4,tetragonal,88,Intermetallic,0.0000,148.99
4712,mp-194651,Ag2ScMn2Se3,hexagonal,177,Chalcogenide,1.0945,105.53


Duplicated ids whose two rows have DIFFERENT formulas: 21 of 21
DS1: 498 rows repeat a formula (237 formulas). Median coefficient of variation of K across same-formula rows: 0.50


## 6 · Rule-based data-quality audit
Each rule is tagged **hard** (impossible by definition), **soft** (outside the documented *typical* range),
**consistency** (internal logical/physical contradiction) or **info** (derived relationship / exposure indicator, not an error).

In [8]:
rules, masks = A.run_rules(dfs)
assert (rules.error == "").all(), rules[rules.error != ""]
rules.drop(columns=[]).to_csv(TAB/"data_quality_rules.csv", index=False)
display(rules[rules.n_violations > 0].sort_values(["dataset","rule_id"])[["rule_id","kind","description","source","n_violations","pct_violations"]])
print("Rules evaluated:", len(rules), "| rules with zero violations:", int((rules.n_violations == 0).sum()))
zero = rules[rules.n_violations == 0]; print("Pass (0 violations):", ", ".join(zero.rule_id))

,rule_id,kind,description,source,n_violations,pct_violations
0,DS1-01,hard,"material_id not unique (same id, conflicting records)",Data dictionary: 'Unique material identifier',42,0.764
1,DS1-02,soft,"formation_energy outside [-5, +2] eV/atom",Data dictionary valid range,126,2.291
3,DS1-04,soft,bulk modulus K outside 'typical' 1-450 GPa,Data dictionary ('typically 1-450'),10,0.182
4,DS1-05,soft,shear modulus G outside 'typical' 0.5-250 GPa,Data dictionary ('typically 0.5-250'),51,0.927
8,DS1-09,consistency,is_stable==1 but formation_energy >= 0,README: 'formation_energy < 0 for stable compounds',369,6.709
9,DS1-10,consistency,"|nu - nu(K,G)| > 0.01 (isotropic elastic identity violated)",README: E=2G(1+nu)=3K(1-2nu),1582,28.764
10,DS1-11,consistency,"Young's modulus from (G,nu) vs from (K,nu) differ by >5%",README: E=2G(1+nu)=3K(1-2nu),1840,33.455
21,DS2-05,soft,|daily_return| > 0.10,Data dictionary ('typically -0.1 to +0.1'),2,0.009
27,DS3-01,hard,condition_rating outside 1-9,README: NBI 1-9,5,0.100
35,DS3-09,info,age_years > design_life_years (beyond design life),Exposure indicator (not an error),654,13.080


Rules evaluated: 55 | rules with zero violations: 40
Pass (0 violations): DS1-03, DS1-06, DS1-07, DS1-08, DS1-12, DS1-13, DS1-14, DS1-15, DS1-16, DS1-17, DS2-01, DS2-02, DS2-03, DS2-04, DS2-06, DS2-07, DS2-08, DS2-09, DS2-10, DS3-02, DS3-03, DS3-04, DS3-05, DS3-06, DS3-07, DS3-08, DS3-12, DS3-13, DS4-01, DS4-02, DS4-03, DS4-04, DS4-05, DS4-06, DS5-01, DS5-03, DS6-01, DS6-03, DS6-05, DS6-06


In [9]:
show = rules[(rules.n_violations > 0) & (rules.kind != "info")].sort_values("pct_violations")
fig, ax = plt.subplots(figsize=(9, 0.38*len(show)+1.2))
col = {"hard":"#c0392b","soft":"#e6a700","consistency":"#2e75b6"}
ax.barh(show.rule_id + "  " + show.description.str.slice(0, 60), show.pct_violations, color=[col[k] for k in show.kind])
ax.set_xlabel("% of dataset rows violating"); ax.set_title("Data-quality rule violations (red=hard, amber=soft, blue=consistency)")
plt.tight_layout(); fig.savefig(FIG/"dq_rule_violations.png", dpi=130); plt.close(fig)
display(HTML(f"<img src='../outputs/figures/dq_rule_violations.png' width=800>")) if False else print("saved dq_rule_violations.png")

saved dq_rule_violations.png


In [10]:
def ex(ds, rid, cols, n=4):
    m = masks[rid]; print(f"\n[{rid}] {rules.set_index('rule_id').description[rid]}  -> {int(m.sum())} rows"); display(dfs[ds].loc[m, cols].head(n))
ex("DS1","DS1-09",["material_id","formula","formation_energy_per_atom_eV","energy_above_hull_eV","is_stable"])
ex("DS1","DS1-10",["material_id","bulk_modulus_GPa","shear_modulus_GPa","poisson_ratio"])
ex("DS3","DS3-01",["bridge_id","material","condition_rating","structurally_deficient"])
ex("DS3","DS3-11",["bridge_id","year_built","age_years","last_inspection_date"])
ex("DS6","DS6-02",["event_id","severity","replacement_value_USD","repair_cost_USD","loss_ratio"])
ex("DS6","DS6-04",["event_id","was_predicted","warning_lead_months","detected_by"])
d1 = dfs["DS1"]; K,G,nu = d1.bulk_modulus_GPa, d1.shear_modulus_GPa, d1.poisson_ratio
dev = (nu - A._poisson_from_KG(K, G)).abs()
print(f"\nElastic identity deviation |nu - nu(K,G)|: mean={dev.mean():.4f} median={dev.median():.4f} p95={dev.quantile(.95):.4f} max={dev.max():.4f}")


[DS1-09] is_stable==1 but formation_energy >= 0  -> 369 rows


,material_id,formula,formation_energy_per_atom_eV,energy_above_hull_eV,is_stable
9,mp-665221,La3Sn4Cu2Nb3,0.1697,0.0334,1
20,mp-128233,BZrMo3,0.0975,0.0405,1
23,mp-885692,Bi3,0.0000,0.0229,1
34,mp-356993,LiIr3Mo3,0.6178,0.0206,1



[DS1-10] |nu - nu(K,G)| > 0.01 (isotropic elastic identity violated)  -> 1582 rows


,material_id,bulk_modulus_GPa,shear_modulus_GPa,poisson_ratio
5,mp-55610,89.35,42.88,0.2737
8,mp-755196,389.79,260.14,0.2393
9,mp-665221,290.24,163.27,0.2511
12,mp-509931,89.68,64.02,0.1882



[DS3-01] condition_rating outside 1-9  -> 5 rows


,bridge_id,material,condition_rating,structurally_deficient
507,BR-00508,Steel (A36),0,1
1097,BR-01098,Steel (A36),0,1
2629,BR-02630,Steel (A36),0,1
2898,BR-02899,Steel (A36),0,1


... (3 outputs left out)

,event_id,severity,replacement_value_USD,repair_cost_USD,loss_ratio
1,FE-00002,Catastrophic,2766322.0,3998388.0,1.4454
11,FE-00012,Catastrophic,2373004.0,3018091.0,1.2718
12,FE-00013,Catastrophic,2108117.0,2740437.0,1.2999
38,FE-00039,Catastrophic,670753.0,912338.0,1.3602



[DS6-04] was_predicted==1 but warning_lead_months==0  -> 51 rows


,event_id,was_predicted,warning_lead_months,detected_by
85,FE-00086,1,0,Routine inspection
106,FE-00107,1,0,Routine inspection
134,FE-00135,1,0,Public report
147,FE-00148,1,0,Load test



Elastic identity deviation |nu - nu(K,G)|: mean=0.0076 median=0.0061 p95=0.0197 max=0.0596


## 7 · Missingness structure
Are the NaNs random, or the warm-up of rolling windows? (`only_leading_rows=True` means NaNs occur only in the first *k* rows of each commodity.)

In [11]:
for k in ["DS2", "DS4"]:
    print(k); display(A.leading_nan_pattern(dfs[k], "commodity", "date"))
print("All other datasets: total missing cells =", {k: int(v.isna().sum().sum()) for k, v in dfs.items() if k not in ("DS2","DS4")})
nm = missing_all[missing_all.missing > 0]
fig, ax = plt.subplots(figsize=(8, 0.3*len(nm)+1.2)); ax.barh(nm.dataset + "." + nm.column, nm.missing_pct, color="#7f7f7f")
ax.set_xlabel("% missing"); ax.set_title("Missing values (all are rolling-window warm-up if table above says so)")
plt.tight_layout(); fig.savefig(FIG/"missingness.png", dpi=130); plt.close(fig); print("saved missingness.png")

DS2


,column,total_nan,nan_per_group_min,nan_per_group_max,only_leading_rows
0,daily_return,8,1,1,True
1,return_5d,40,5,5,True
2,return_21d,168,21,21,True
3,volatility_5d_ann,40,5,5,True
4,volatility_21d_ann,168,21,21,True
5,volatility_63d_ann,504,63,63,True
6,sma_21,160,20,20,True
7,sma_63,496,62,62,True
8,bollinger_upper,160,20,20,True
9,bollinger_lower,160,20,20,True


DS4


,column,total_nan,nan_per_group_min,nan_per_group_max,only_leading_rows
0,mqi_5d_trend,40,5,5,True
1,mqi_21d_trend,168,21,21,True
2,mqi_63d_trend,504,63,63,True


All other datasets: total missing cells = {'DS1': 0, 'DS3': 0, 'DS5': 0, 'DS6': 0}


saved missingness.png


## 8 · Outliers (3×IQR fences; flagged, never removed)

In [12]:
out_all = []
for k, df in dfs.items():
    t = A.outlier_table(df); t.insert(0, "dataset", k); out_all.append(t)
out_all = pd.concat(out_all); out_all.to_csv(TAB/"outliers.csv", index=False)
display(out_all[out_all.n_extreme > 0].sort_values("pct_extreme", ascending=False).head(25))

,dataset,column,lower_fence,upper_fence,n_extreme,pct_extreme,skew,kurtosis
18,DS2,macd_signal,-1.262881e+01,1.251902e+01,6021,26.233,1.576700,26.005211
17,DS2,macd,-1.345879e+01,1.335002e+01,5993,26.111,1.499507,26.125571
0,DS5,price_usd_per_kg,-4.962900e+02,6.723600e+02,792,15.000,4.671604,21.301717
1,DS5,base_price,-5.050000e+02,6.850000e+02,792,15.000,4.628087,20.802600
14,DS2,bollinger_lower,-3.113383e+03,4.258211e+03,2849,12.500,2.447156,4.508242
13,DS2,bollinger_upper,-3.474833e+03,4.761215e+03,2849,12.500,2.438769,4.429680
2,DS2,low,-3.296154e+03,4.511428e+03,2869,12.500,2.447158,4.498126
3,DS2,close,-3.317595e+03,4.540720e+03,2869,12.500,2.447059,4.497474
11,DS2,sma_21,-3.284175e+03,4.496697e+03,2849,12.500,2.436589,4.422353
12,DS2,sma_63,-3.215346e+03,4.406335e+03,2807,12.500,2.413971,4.258603


## 9 · Temporal structure

In [13]:
d2, d4, d5, d6, d3 = dfs["DS2"], dfs["DS4"], dfs["DS5"], dfs["DS6"], dfs["DS3"]
bd = pd.bdate_range(d2.date.min(), d2.date.max())
t = d2.groupby("commodity").agg(rows=("date","size"), start=("date","min"), end=("date","max"))
t["weekdays_in_span"] = len(bd); t["weekdays_missing"] = t.weekdays_in_span - t.rows
display(t)
us_hol = [pd.Timestamp(x) for x in ["2014-12-25","2015-12-25","2016-12-26","2019-07-04","2020-01-01","2021-12-24","2022-12-26","2023-01-02"]]
print("Common US exchange holidays present as trading days in DS2:", [str(h.date()) for h in us_hol if h in set(d2.date)])
gaps = d2.sort_values("date").groupby("commodity").date.diff().dt.days
print("Gap distribution (days):", gaps.value_counts().sort_index().to_dict())
print("DS2 and DS4 calendars identical:", set(map(tuple, d2[["date","commodity"]].values)) == set(map(tuple, d4[["date","commodity"]].values)))
print("\nDS5 months per element:", d5.groupby("element").size().unique(), "| date span:", d5.date.min().date(), "->", d5.date.max().date(), "| month gaps:", int((d5.sort_values('date').groupby('element').date.diff().dt.days.dropna() > 31).sum()))
print("DS6 events per year:", d6.event_year.value_counts().sort_index().to_dict())
print("DS3 snapshot: year_built+age is constant =", sorted((dfs['DS3'].year_built + dfs['DS3'].age_years).unique()), "| last_inspection range:", d3.last_inspection_date.min().date(), "->", d3.last_inspection_date.max().date())

,rows,start,end,weekdays_in_span,weekdays_missing
commodity,,,,,
Aluminium,2869,2014-01-02,2024-12-31,2869,0
Cobalt_Index,2869,2014-01-02,2024-12-31,2869,0
Copper,2869,2014-01-02,2024-12-31,2869,0
Iron_Ore,2869,2014-01-02,2024-12-31,2869,0
Lithium_ETF,2869,2014-01-02,2024-12-31,2869,0
Nickel,2869,2014-01-02,2024-12-31,2869,0
RareEarth_ETF,2869,2014-01-02,2024-12-31,2869,0
Steel_HRC,2869,2014-01-02,2024-12-31,2869,0


Common US exchange holidays present as trading days in DS2: ['2014-12-25', '2015-12-25', '2016-12-26', '2019-07-04', '2020-01-01', '2021-12-24', '2022-12-26', '2023-01-02']
Gap distribution (days): {1.0: 18352, 3.0: 4592}
DS2 and DS4 calendars identical: True

DS5 months per element: [132] | date span: 2014-01-01 -> 2024-12-01 | month gaps: 0
DS6 events per year: {2005: 97, 2006: 111, 2007: 104, 2008: 97, 2009: 128, 2010: 94, 2011: 108, 2012: 95, 2013: 98, 2014: 104, 2015: 83, 2016: 107, 2017: 111, 2018: 95, 2019: 109, 2020: 114, 2021: 105, 2022: 124, 2023: 116}
DS3 snapshot: year_built+age is constant = [np.int64(2024)] | last_inspection range: 2020-01-01 -> 2023-12-28


In [14]:
fig, ax = plt.subplots(figsize=(10, 2.8))
spans = [("DS2/DS4 daily prices+MQI", d2.date.min(), d2.date.max()), ("DS5 monthly elements", d5.date.min(), d5.date.max()),
         ("DS3 last_inspection_date", d3.last_inspection_date.min(), d3.last_inspection_date.max()),
         ("DS6 failure events", pd.Timestamp(int(d6.event_year.min()),1,1), pd.Timestamp(int(d6.event_year.max()),12,31))]
for i, (n, a, b) in enumerate(spans): ax.barh(i, (b-a).days, left=matplotlib.dates.date2num(a), height=0.5)
ax.set_yticks(range(len(spans))); ax.set_yticklabels([s[0] for s in spans]); ax.xaxis_date(); ax.set_title("Temporal coverage: market data 2014-2024 vs failures 2005-2023 vs inspections 2020-2023")
plt.tight_layout(); fig.savefig(FIG/"temporal_coverage.png", dpi=130); plt.close(fig); print("saved temporal_coverage.png")

saved temporal_coverage.png


## 10 · DS5 `monthly_return` and price resolution
The README lists `monthly_return` as a key DS5 column. Is it consistent with the supplied prices?

In [15]:
s = d5.sort_values(["element","date"]).copy(); gp = s.groupby("element").price_usd_per_kg
cand = {"simple return t/t-1": gp.pct_change(), "log return": np.log(gp.transform(lambda x: x/x.shift())), "price/base_price-1": s.price_usd_per_kg/s.base_price-1}
display(pd.DataFrame({k: {"corr with monthly_return": v.corr(s.monthly_return), "max |diff|": (v - s.monthly_return).abs().max()} for k, v in cand.items()}).T)
print("monthly_return: mean=%.4f std=%.4f min=%.3f max=%.3f  (std of price-derived monthly return: %.4f)" % (s.monthly_return.mean(), s.monthly_return.std(), s.monthly_return.min(), s.monthly_return.max(), cand["simple return t/t-1"].std()))
unchanged = s.assign(same=(gp.diff()==0)).groupby("element").same.mean().sort_values(ascending=False)
print("Share of months with an UNCHANGED price (price resolution issue), top 8:"); display(unchanged.head(8).round(3).to_frame("share_unchanged"))
print("Per-element price range (max/min):"); display((s.groupby("element").price_usd_per_kg.agg(["min","max"]).assign(ratio=lambda x: x["max"]/x["min"])).sort_values("ratio", ascending=False).head(5))

,corr with monthly_return,max |diff|
simple return t/t-1,-0.000476,1.402532
log return,-0.000052,1.402373
price/base_price-1,0.004920,1.281200


monthly_return: mean=0.0031 std=0.2364 min=-1.385 max=1.259  (std of price-derived monthly return: 0.0130)
Share of months with an UNCHANGED price (price resolution issue), top 8:


,share_unchanged
element,
Fe,0.924
N,0.765
C,0.636
Al,0.303
Pb,0.227
Mn,0.182
Mg,0.144
Zn,0.114


Per-element price range (max/min):


,min,max,ratio
element,,,
Nd,74.10,100.17,1.351822
La,4.05,5.45,1.345679
Ce,4.08,5.44,1.333333
Dy,297.20,373.57,1.256965
Bi,8.93,11.15,1.248600


## 11 · Category alignment across datasets

In [16]:
align = pd.DataFrame([
 {"pair":"DS2.commodity vs DS4.commodity","identical_label_sets": set(d2.commodity)==set(d4.commodity), "n": d2.commodity.nunique()},
 {"pair":"DS3.material vs DS6.material","identical_label_sets": set(d3.material)==set(d6.material), "n": d3.material.nunique()},
 {"pair":"DS3.location vs DS6.location","identical_label_sets": set(d3.location)==set(d6.location), "n": d3.location.nunique()},
 {"pair":"DS3.corrosion_environment vs DS6.corrosion_environment","identical_label_sets": set(d3.corrosion_environment)==set(d6.corrosion_environment), "n": d3.corrosion_environment.nunique()},
]); display(align)
print("DS3 bridge_type vs DS6 structure_type label overlap:", sorted(set(d3.bridge_type) & set(d6.structure_type)), "(DS6 spans Bridge/Building/Pipeline/Power Plant; DS3 is bridges only)")
print("Bridge-only DS6 events:", int((d6.structure_type=='Bridge').sum()), "of", len(d6))

,pair,identical_label_sets,n
0,DS2.commodity vs DS4.commodity,True,8
1,DS3.material vs DS6.material,True,11
2,DS3.location vs DS6.location,True,15
3,DS3.corrosion_environment vs DS6.corrosion_environment,True,3


DS3 bridge_type vs DS6 structure_type label overlap: [] (DS6 spans Bridge/Building/Pipeline/Power Plant; DS3 is bridges only)
Bridge-only DS6 events: 1011 of 2000


## 12 · Workbook (xlsx) vs CSV files

In [17]:
for k, v in xl.items():
    if k.startswith(("DS5_Stress","DS3_Infra","DS1_Material_Properties::VALIDATION")):
        print("="*100); print(k); display(v)
snap = next(v for k, v in xl.items() if k.startswith("DS4_Cross") and "MQI (0-100)" in v.columns)
print("\nxlsx DS4 'daily snapshot as of 2024-01-15' vs DS4 CSV on 2024-01-15:")
csv15 = d4[d4.date == "2024-01-15"].set_index("commodity").mqi
print(pd.DataFrame({"xlsx_MQI": snap["MQI (0-100)"].values, "csv_MQI": csv15.reindex(["Steel_HRC","Aluminium","Copper","Nickel","Lithium_ETF","Cobalt_Index","RareEarth_ETF","Iron_Ore"]).values}, index=snap.Commodity.values))
s1 = next(v for k, v in xl.items() if k.startswith("DS1_Material_Properties::block1"))
print("\nxlsx DS1 sample ids found in DS1 CSV:", int(s1["Material ID"].isin(dfs["DS1"].material_id).sum()), "/", len(s1))

DS1_Material_Properties::VALIDATION CHECKS


,E = 2G(1+ν),Computed E (GPa),col2,col3,col4,col5
0,Fe,211.560,E from K,214.20,Consistency %,1.247873
1,Al,70.200,E from K,68.40,Consistency %,2.564103
2,Cu,128.640,E from K,134.40,Consistency %,4.477612
3,Ti,116.160,E from K,118.80,Consistency %,2.272727
4,Si,161.040,E from K,164.64,Consistency %,2.235469
5,Ni,199.120,E from K,205.20,Consistency %,3.053435
6,W,412.160,E from K,409.20,Consistency %,0.718168
7,Fe2O3,228.600,E from K,245.64,Consistency %,7.454068
8,Al2O3,404.240,E from K,396.24,Consistency %,1.979022
9,TiO2,281.600,E from K,290.40,Consistency %,3.125000


DS3_Infrastructure_Portfolio::block1


,Asset ID,Asset Type,Sub-Type,Primary Material,Material Grade,Age (years),Design Life (years),Location,Climate Zone,Condition (1-9),Loan Outstanding ($M),Loan Tenor Remaining (yr),Annual Traffic/Usage,Last Inspection,Corrosion Environment,Insurance Premium ($K/yr)
0,BR-001,Bridge,Highway,Structural Steel,ASTM A36,42,75,"Jacksonville, FL",Coastal Subtropical,5,12.0,8,"35,000 ADT",2023-09-15,High Chloride,185
1,BR-002,Bridge,Railroad,Weathering Steel,ASTM A588,28,75,"Cleveland, OH",Industrial Temperate,6,8.5,12,45 trains/day,2023-11-20,Industrial + De-icing,95
2,BR-003,Bridge,Pedestrian,Stainless Steel,316L,15,100,"Chicago, IL",Urban Temperate,7,3.2,15,"5,000 ped/day",2024-01-10,Urban + De-icing,28
3,PL-001,Pipeline,Oil,Carbon Steel,API 5L X65,35,50,"Galveston, TX",Coastal Subtropical,4,45.0,5,250K bbl/day,2023-06-30,Coastal + H2S,520
4,PL-002,Pipeline,Natural Gas,Carbon Steel,API 5L X70,22,50,"Casper, WY",Inland Arid,7,28.0,15,1.2 BCF/day,2024-02-28,Mild (inland),180
5,BL-001,Building,Office Tower,Reinforced Concrete,C40 w/ rebar,38,60,"Miami Beach, FL",Coastal Tropical,5,65.0,10,"2,500 occupants",2023-08-15,Severe Chloride,480
6,BL-002,Building,Warehouse,Pre-Stressed Concrete,C50 PT,25,60,"Savannah, GA",Inland Subtropical,7,15.0,18,50 trucks/day,2024-03-01,Moderate,65
7,PP-001,Power Plant,Gas Turbine,Ni-based Superalloy,Inconel 718,30,40,"Newark, NJ",Industrial Coastal,5,120.0,5,500 MW capacity,2023-12-01,High Temp + Coastal,1250
8,BR-004,Bridge,Highway,Prestressed Concrete,C60 PT w/ tendons,50,75,"Wilmington, NC",Coastal Subtropical,4,18.0,6,"42,000 ADT",2023-07-20,Severe Chloride,245
9,BR-005,Bridge,Highway,Composite Steel+Concrete,A36 + C40,55,75,"Hartford, CT",Urban Temperate,3,22.0,4,"55,000 ADT",2023-10-05,Urban + De-icing + Age,380


DS3_Infrastructure_Portfolio::MATERIAL PROPERTIES FOR PORTFOLIO ASSETS


,Asset ID,Material,Tensile Strength (MPa),Yield Strength (MPa),Corrosion Rate (mm/yr),Paris C,Paris m,Failure Mode,Wall/Cover Thickness (mm),Critical Thickness (mm)
0,BR-001,A36 Steel,450,250,0.080,6.900000e-12,3.0,Ductile (fatigue),18,8
1,BR-002,A588 Weathering,485,345,0.040,5.200000e-12,2.8,Ductile (fatigue),16,7
2,BR-003,316L Stainless,515,205,0.010,3.100000e-12,3.2,Pitting corrosion,12,5
3,PL-001,API 5L X65,535,450,0.120,8.500000e-12,3.1,Stress corrosion cracking,22,10
4,PL-002,API 5L X70,570,485,0.030,7.800000e-12,3.0,Ductile (fatigue),20,9
5,BL-001,C40 Concrete + Rebar,25,20,0.060,NaN,NaN,Punching shear / spalling,50,25
6,BL-002,C50 PT Concrete,35,28,0.020,NaN,NaN,Tendon corrosion,60,30
7,PP-001,Inconel 718,1240,1035,0.005,4.200000e-12,3.5,Creep + fatigue,8,4
8,BR-004,C60 PT + Tendons,40,32,0.090,NaN,NaN,Tendon corrosion / brittle,55,25
9,BR-005,A36 + C40 Composite,400,230,0.070,6.500000e-12,3.0,Combined fatigue + spalling,15,7


DS5_Stress_Scenarios::block1


,Scenario ID,Scenario Name,Shock Type,Shock Magnitude,Shock Distribution,Duration (quarters),Affected Commodities,Affected Assets,Historical Precedent,Calibration Source
0,SC-001,Rare Earth Supply Crisis,Supply restriction,40% export reduction,"Step function + N(0,5%) noise",3-4,"Rare earths, lithium, cobalt",PP-001,China 2010 quota,USGS + Reuters 2010
1,SC-002,Steel Quality Degradation,Quality shock,15% tensile strength drop,Gradual (linear over 2Q),2-3,"Steel HRC, iron ore","BR-001, BR-002, BR-005",Quality issues at major mills,Industry reports
2,SC-003,Climate-Accelerated Corrosion,Environmental,30% corrosion rate increase,Permanent step + seasonal,Permanent,All metals (indirect),"All coastal: BR-001,PL-001,BL-001,BR-004",IPCC AR6 +2°C,IPCC + corrosion lit.
3,SC-004,Recycled Material Mandate,Regulatory,50% recycled content req.,Phase-in (linear 4Q),4-6 (phase-in),"Aluminium, steel, copper",All (compliance cost),EU Circular Economy,EU regulation text
4,SC-005,Novel Alloy Disruption,Technology,"New alloy: 20% stronger, 15% cheaper",S-curve adoption,6-8 (adoption),"Steel, titanium",All steel bridges,Historical alloy transitions,Technology adoption lit.


DS5_Stress_Scenarios::DETAILED SHOCK PARAMETERS FOR MONTE CARLO


,Scenario,Commodity,Price Impact (mean %),Price Impact (σ %),RUL Impact (mean %),RUL Impact (σ %),Insurance Loss Mult.,Recovery Time (Q),Correlation w/ equity
0,SC-001,Rare Earth Index,180,45,-5,2,1.0,8-12,-0.3
1,SC-001,Lithium,25,10,0,0,1.0,4-6,-0.2
2,SC-001,Iron Ore,12,5,0,0,1.0,2-3,0.1
3,SC-002,Steel HRC,-8,4,-15,5,1.8,4-6,0.0
4,SC-002,Iron Ore,-5,3,0,0,1.0,3-4,0.0
5,SC-003,All metals,3,2,-25,8,2.5,Permanent,0.0
6,SC-004,Aluminium (virgin),-15,6,0,0,1.0,6-8,0.1
7,SC-004,Aluminium (recycled),25,8,0,0,1.0,6-8,0.1
8,SC-005,Steel HRC,-20,8,10,5,0.7,8-12,-0.1
9,SC-005,Titanium,-10,5,5,3,0.8,6-8,0.0



xlsx DS4 'daily snapshot as of 2024-01-15' vs DS4 CSV on 2024-01-15:
                  xlsx_MQI  csv_MQI
Steel HRC             78.5    83.74
Aluminium LME         82.1    84.00
Copper COMEX          85.3    85.60
Nickel LME            71.2    70.33
Lithium               68.8    67.06
Cobalt                74.5    76.56
Rare Earth Index      62.3    64.98
Iron Ore              80.0    80.43

xlsx DS1 sample ids found in DS1 CSV: 0 / 50


## 13 · Relationships between datasets (only what the data support)

In [18]:
fk = A.candidate_foreign_keys({k: dfs[k] for k in ["DS1","DS2","DS3","DS4","DS5","DS6"]})
print("Automated value-overlap search for foreign-key candidates (categorical/string/date, or same-named integers):"); display(fk)
rel = A.relationship_evidence(dfs, xl); rel.to_csv(TAB/"relationship_evidence.csv", index=False)
display(rel)

Automated value-overlap search for foreign-key candidates (categorical/string/date, or same-named integers):


,left,right,left_unique,right_unique,intersection,overlap_of_smaller
0,DS2.commodity,DS4.commodity,8,8,8,1.000
1,DS2.date,DS4.date,2869,2869,2869,1.000
2,DS3.corrosion_environment,DS6.corrosion_environment,3,3,3,1.000
3,DS3.material,DS6.material,11,11,11,1.000
4,DS3.location,DS6.location,15,15,15,1.000
5,DS1.formula,DS5.element,5002,40,37,0.925
6,DS2.date,DS3.last_inspection_date,2869,1312,938,0.715
7,DS3.last_inspection_date,DS4.date,1312,2869,938,0.715
8,DS2.date,DS5.date,2869,132,94,0.712
9,DS4.date,DS5.date,2869,132,94,0.712


,relationship,key,cardinality,evidence,supported
0,DS2 <-> DS4,"(date, commodity)",1:1,"matched=22952, DS2-only=0, DS4-only=0; duplicate keys DS2=0, DS4=0",YES - exact key join
1,DS3 <-> DS6 on material,material,many:many (no asset id in DS6),"DS6 rows whose material exists in DS3: 100.0%; DS3 labels=11, DS6 labels=11",PARTIAL - categorical (segment-level) link only; NOT asset-level
2,DS3 <-> DS6 on location,location,many:many (no asset id in DS6),"DS6 rows whose location exists in DS3: 100.0%; DS3 labels=15, DS6 labels=15",PARTIAL - categorical (segment-level) link only; NOT asset-level
3,DS3 <-> DS6 on corrosion_environment,corrosion_environment,many:many (no asset id in DS6),"DS6 rows whose corrosion_environment exists in DS3: 100.0%; DS3 labels=3, DS...",PARTIAL - categorical (segment-level) link only; NOT asset-level
4,DS3 <-> DS6 asset-level,none,n/a,DS6 has event_id only; DS3 has bridge_id only; no shared asset identifier,NO
5,DS1 <-> DS5,elements parsed from DS1.formula <-> DS5.element,1:many (material -> its elements),DS1 materials with ALL elements priced in DS5: 32.6%; with >=1 priced elemen...,PARTIAL - derived key; only for materials whose elements are all priced
6,DS1 <-> DS3/DS6 (material),DS3/DS6.material vs DS1.formula/category,n/a,exact label matches between DS3/DS6 material names and DS1 formula/category: 0,NO - engineering material grades (e.g. 'Steel (A36)') are not DS1 compounds;...
7,DS2 <-> DS5 (commodity vs element),semantic crosswalk (e.g. Copper->Cu),n/a,"monthly-return correlations: Aluminium-Al: -0.06, Copper-Cu: -0.10, Nickel-N...",NO key; crosswalk possible but data show no co-movement
8,xlsx portfolio assets <-> DS3 CSV,Asset ID vs bridge_id,n/a,xlsx ids e.g. BR-001 vs CSV ids e.g. BR-00001; exact matches=0,NO - separate fictional 10-asset simulator portfolio (per xlsx note)
9,DS6 <-> DS2/DS4/DS5 (time),event_year vs calendar year,many:many,DS6 years 2005-2023 vs market data 2014-2024,ONLY temporal context (no asset/commodity key)


## 14 · Statistical-structure preview (NOT modelling)
Before designing models we check how much signal the data can possibly carry, so that later results are interpreted honestly.

In [19]:
sp = dfs["DS1"][["formation_energy_per_atom_eV","energy_above_hull_eV","band_gap_eV","bulk_modulus_GPa","shear_modulus_GPa","poisson_ratio","density_g_cm3","melting_point_K"]].corr(method="spearman").round(2)
display(sp)
piv = d2.pivot(index="date", columns="commodity", values="daily_return").dropna(); cm = piv.corr()
print("DS2 cross-commodity daily-return correlation: mean off-diagonal = %.3f, max = %.3f, min = %.3f" % ((cm.values.sum()-len(cm))/(len(cm)**2-len(cm)), cm.values[np.triu_indices(len(cm),1)].max(), cm.values[np.triu_indices(len(cm),1)].min()))
m = d2.merge(d4, on=["date","commodity"]).sort_values(["commodity","date"])
m["fwd_21d_logret"] = m.groupby("commodity").daily_return.transform(lambda s: s[::-1].rolling(21).sum()[::-1].shift(-1))
print("corr(MQI 21d trend, FORWARD 21d log-return) pooled = %.4f ; corr(MQI level, same-day return) = %.4f" % (m.mqi_21d_trend.corr(m.fwd_21d_logret), m.mqi.corr(m.daily_return)))
ac = piv.apply(lambda s: s.autocorr(1)); print("DS2 lag-1 return autocorrelation by commodity:", ac.round(3).to_dict())
from scipy import stats
print("DS2 return tail check (excess kurtosis):", piv.apply(lambda s: stats.kurtosis(s)).round(2).to_dict())
d3x = d3.assign(thick_frac=d3.remaining_thickness_mm/d3.original_thickness_mm)
print("DS3 Spearman with condition_rating:", d3x[["age_years","corrosion_rate_mm_yr","thick_frac","fatigue_cycles_millions","adt"]].corrwith(d3x.condition_rating, method="spearman").round(3).to_dict())
a3 = d3.age_years; a6 = d6.age_at_event_years
print("Age distributions  DS3 (snapshot population): mean %.1f sd %.1f | DS6 (age at failure): mean %.1f sd %.1f | KS-stat %.3f" % (a3.mean(), a3.std(), a6.mean(), a6.std(), stats.ks_2samp(a3, a6).statistic))
print("Mean corrosion rate by environment  DS3 vs DS6:"); display(pd.concat([d3.groupby('corrosion_environment').corrosion_rate_mm_yr.mean(), d6.groupby('corrosion_environment').corrosion_rate_mm_yr.mean()], axis=1, keys=["DS3 (population)","DS6 (failures)"]).round(4))

,formation_energy_per_atom_eV,energy_above_hull_eV,band_gap_eV,bulk_modulus_GPa,shear_modulus_GPa,poisson_ratio,density_g_cm3,melting_point_K
formation_energy_per_atom_eV,1.00,0.39,-0.57,-0.02,-0.01,0.00,0.18,-0.25
energy_above_hull_eV,0.39,1.00,-0.27,-0.01,-0.01,-0.01,0.08,-0.14
band_gap_eV,-0.57,-0.27,1.00,0.01,0.01,0.00,-0.36,0.21
bulk_modulus_GPa,-0.02,-0.01,0.01,1.00,0.87,-0.01,0.12,0.24
shear_modulus_GPa,-0.01,-0.01,0.01,0.87,1.00,-0.45,0.09,0.21
poisson_ratio,0.00,-0.01,0.00,-0.01,-0.45,1.00,0.01,-0.01
density_g_cm3,0.18,0.08,-0.36,0.12,0.09,0.01,1.00,0.01
melting_point_K,-0.25,-0.14,0.21,0.24,0.21,-0.01,0.01,1.00


DS2 cross-commodity daily-return correlation: mean off-diagonal = 0.008, max = 0.038, min = -0.019
corr(MQI 21d trend, FORWARD 21d log-return) pooled = 0.0130 ; corr(MQI level, same-day return) = 0.0023
DS2 lag-1 return autocorrelation by commodity: {'Aluminium': 0.018, 'Cobalt_Index': 0.005, 'Copper': -0.006, 'Iron_Ore': -0.007, 'Lithium_ETF': 0.018, 'Nickel': 0.024, 'RareEarth_ETF': 0.021, 'Steel_HRC': -0.012}


DS2 return tail check (excess kurtosis): {'Aluminium': -0.04, 'Cobalt_Index': 0.01, 'Copper': -0.08, 'Iron_Ore': 0.13, 'Lithium_ETF': 0.14, 'Nickel': -0.14, 'RareEarth_ETF': 0.01, 'Steel_HRC': 0.01}
DS3 Spearman with condition_rating: {'age_years': -0.432, 'corrosion_rate_mm_yr': -0.703, 'thick_frac': 0.689, 'fatigue_cycles_millions': -0.159, 'adt': -0.011}
Age distributions  DS3 (snapshot population): mean 46.6 sd 17.9 | DS6 (age at failure): mean 45.3 sd 15.1 | KS-stat 0.073
Mean corrosion rate by environment  DS3 vs DS6:


,DS3 (population),DS6 (failures)
corrosion_environment,,
Mild,0.0232,0.0290
Moderate,0.0446,0.0583
Severe,0.0668,0.0804


## 15 · Candidate targets, identifiers and leakage-risk columns
*Analyst judgement, each tied to evidence above and to the README purposes.*

In [20]:
targets = pd.DataFrame([
 ("DS1","bulk_modulus_GPa, shear_modulus_GPa","regression target","README: CGNN/PINN/ensemble training; physics: K,G>0. Derive nu,E from (K,G) to enforce elastic identity."),
 ("DS1","formation_energy_per_atom_eV, band_gap_eV, density_g_cm3, melting_point_K","regression targets","Named in README as key property columns."),
 ("DS1","poisson_ratio","derived (do not model independently)","Only ~71% of rows satisfy nu=nu(K,G) within 0.01 (rule DS1-10)."),
 ("DS1","is_stable, is_metal","classification targets (leakage-prone)","is_stable: 369 rows contradict README (Ef>=0) -> use with caution; is_metal == (band_gap==0) exactly -> never use band_gap to predict is_metal."),
 ("DS1","material_id","identifier - NOT unique","42 rows share 21 ids with conflicting formulas (rule DS1-01) -> use row position as key."),
 ("DS2","close / daily_return (next-day or h-day forward)","forecast / risk target","Need forward targets built from close; all supplied indicators are backward-looking."),
 ("DS2","(date, commodity)","composite primary key","Unique; joins 1:1 to DS4."),
 ("DS3","condition_rating (NBI 1-9)","ordinal target / PD proxy input","Strongly related to corrosion and thickness fraction (see sec.14)."),
 ("DS3","structurally_deficient","derived label","Exactly equals (condition_rating<=4) (rule DS3-08) -> leakage if rating-derived features used alongside."),
 ("DS3","remaining_thickness_mm","derived","Equals original - corrosion_rate*age (rule DS3-10) -> redundant with corrosion*age."),
 ("DS3","loan_outstanding_M, replacement_cost_M, insurance_premium_K_yr","exposure (EAD) inputs","Financial quantities that actually exist in the data."),
 ("DS3","bridge_id","primary key","Unique; no counterpart in DS6."),
 ("DS4","mqi, mqi_*d_trend, supply_disruption_prob, ...","features aligned to DS2","Backward-looking (rule DS4-06); safe at time t."),
 ("DS5","price_usd_per_kg","element price series","`monthly_return` irreconcilable with price (sec.10) -> recompute returns from price."),
 ("DS6","severity, loss_ratio, repair_cost_USD","outcome / LGD source","Only dataset with realised loss severity (loss_ratio = repair/replacement)."),
 ("DS6","was_predicted, warning_lead_months, detected_by, insurance_claim_filed, regulatory_action","POST-EVENT outcomes","Known only after failure -> exclude from any failure-prediction feature set."),
 ("DS6","(all rows)","structural limitation","Contains ONLY failure events (no survivors / censored units) -> hazard cannot be estimated from DS6 alone."),
], columns=["dataset","columns","role","evidence / note"])
targets.to_csv(TAB/"candidate_targets.csv", index=False); display(targets)

,dataset,columns,role,evidence / note
0,DS1,"bulk_modulus_GPa, shear_modulus_GPa",regression target,"README: CGNN/PINN/ensemble training; physics: K,G>0. Derive nu,E from (K,G) ..."
1,DS1,"formation_energy_per_atom_eV, band_gap_eV, density_g_cm3, melting_point_K",regression targets,Named in README as key property columns.
2,DS1,poisson_ratio,derived (do not model independently),"Only ~71% of rows satisfy nu=nu(K,G) within 0.01 (rule DS1-10)."
3,DS1,"is_stable, is_metal",classification targets (leakage-prone),is_stable: 369 rows contradict README (Ef>=0) -> use with caution; is_metal ...
4,DS1,material_id,identifier - NOT unique,42 rows share 21 ids with conflicting formulas (rule DS1-01) -> use row posi...
5,DS2,close / daily_return (next-day or h-day forward),forecast / risk target,Need forward targets built from close; all supplied indicators are backward-...
6,DS2,"(date, commodity)",composite primary key,Unique; joins 1:1 to DS4.
7,DS3,condition_rating (NBI 1-9),ordinal target / PD proxy input,Strongly related to corrosion and thickness fraction (see sec.14).
8,DS3,structurally_deficient,derived label,Exactly equals (condition_rating<=4) (rule DS3-08) -> leakage if rating-deri...
9,DS3,remaining_thickness_mm,derived,Equals original - corrosion_rate*age (rule DS3-10) -> redundant with corrosi...


## 16 · Final dataset summary table

In [21]:
struct = {"DS1":"cross-section (independent observations)", "DS2":"panel: 8 commodities x ~2,869 business days", "DS3":"cross-section snapshot (as of 2024; age = 2024 - year_built)",
          "DS4":"panel, same (date, commodity) grid as DS2", "DS5":"panel: 40 elements x 132 months", "DS6":"event log (failures only), 2005-2023"}
rows = []
for k, df in dfs.items():
    dc = A.DATE_COLS[k]; rng = f"{df[dc[0]].min().date()} -> {df[dc[0]].max().date()}" if dc else ("event_year %d-%d" % (df.event_year.min(), df.event_year.max()) if k=="DS6" else "-")
    p = profiles[k]
    rows.append({"dataset":k, "file":A.CSV_FILES[k], "rows":len(df), "cols":df.shape[1], "structure":struct[k], "date_range":rng,
                 "missing_cells":int(df.isna().sum().sum()), "full_dup_rows":int(df.duplicated().sum()),
                 "candidate_key":str(A.candidate_keys(df)[:2]), "constant_cols":int(p.constant.sum()), "near_constant_cols":int(p.near_constant.sum()),
                 "rule_violations(non-info)":int(rules[(rules.dataset==k)&(rules.kind!="info")].n_violations.sum()),
                 "rules_failed":", ".join(rules[(rules.dataset==k)&(rules.kind!="info")&(rules.n_violations>0)].rule_id)})
summary = pd.DataFrame(rows); summary.to_csv(TAB/"dataset_summary.csv", index=False); display(summary)

,dataset,file,rows,cols,structure,date_range,missing_cells,full_dup_rows,candidate_key,constant_cols,near_constant_cols,rule_violations(non-info),rules_failed
0,DS1,DS1_material_properties_5500.csv,5500,18,cross-section (independent observations),-,0,0,"[('material_id', 'formula'), ('material_id', 'spacegroup_number')]",0,0,4020,"DS1-01, DS1-02, DS1-04, DS1-05, DS1-09, DS1-10, DS1-11"
1,DS2,DS2_commodity_prices_10yr.csv,22952,24,"panel: 8 commodities x ~2,869 business days",2014-01-02 -> 2024-12-31,2760,0,"[('date', 'commodity')]",0,0,2,DS2-05
2,DS3,DS3_infrastructure_bridges_5000.csv,5000,24,cross-section snapshot (as of 2024; age = 2024 - year_built),2020-01-01 -> 2023-12-28,0,0,"[('bridge_id',)]",0,0,28,"DS3-01, DS3-11"
3,DS4,DS4_crossdomain_features_daily.csv,22952,12,"panel, same (date, commodity) grid as DS2",2014-01-02 -> 2024-12-31,712,0,"[('date', 'commodity')]",0,0,0,
4,DS5,DS5_element_prices_monthly.csv,5280,5,panel: 40 elements x 132 months,2014-01-01 -> 2024-12-01,0,0,"[('date', 'element')]",0,0,5214,DS5-02
5,DS6,DS6_historical_failures_2000.csv,2000,18,"event log (failures only), 2005-2023",event_year 2005-2023,0,0,"[('event_id',)]",0,0,136,"DS6-02, DS6-04"


## 17 · Cleaning decisions: PROBLEM → DECISION → REASON

In [22]:
R = rules.set_index("rule_id").n_violations
d5_mismatch = int(R["DS5-02"]); n5 = len(d5)
decisions = pd.DataFrame([
 ("DS1", f"material_id duplicated: {R['DS1-01']} rows / {dup.material_id.nunique()} ids with conflicting formulas",
  "Keep all rows; use positional row_id as the key; never join on material_id",
  "Cannot tell which record is correct; dropping either would be arbitrary. Joins on material_id would silently multiply rows."),
 ("DS1", f"{d1.formula.duplicated().sum()} repeated formulas with large property spread (median K CV {(g.K_std/g.K_mean).median():.2f})",
  "Keep; use GroupKFold by formula in all CV",
  "Same-formula rows in train and test would leak; rows are not interchangeable duplicates."),
 ("DS1", f"is_stable==1 with formation_energy>=0: {R['DS1-09']} rows (README says stable => Ef<0)",
  "Keep; do not treat is_stable as a clean ground truth; report model skill against it with this caveat",
  "Label contradicts the documented definition for 6.7% of rows; cannot repair without inventing a rule."),
 ("DS1", f"Elastic identity violated: nu vs nu(K,G) >0.01 in {R['DS1-10']} rows ({100*R['DS1-10']/len(d1):.1f}%); E(G,nu) vs E(K,nu) >5% in {R['DS1-11']} rows",
  "Keep; model K and G as primary targets and DERIVE nu and E from them so predictions satisfy the identity by construction",
  "README demands E=2G(1+nu)=3K(1-2nu); supplied triplets are mutually inconsistent, so at most two of (K,G,nu) can be trusted."),
 ("DS1", f"Values beyond documented 'typical' ranges: Ef<-5 in {R['DS1-02']} rows, K>450 in {R['DS1-04']}, G>250 in {R['DS1-05']}",
  "Keep; flag; use robust/transform-aware modelling", "Soft ranges ('typically'); high-K/G (e.g. carbides, borides) are physically possible. Removal would bias the stiff-material tail."),
 ("DS2", f"NaNs in rolling indicators (max {int(missing_all[missing_all.dataset=='DS2'].missing.max())} per column)",
  "Keep as NaN; no imputation; drop warm-up rows only inside feature/target windows", "Section 7 shows NaNs exist only in the first k rows of each commodity (rolling-window warm-up), so they carry no information to impute."),
 ("DS2", f"|daily_return|>10% in {R['DS2-05']} rows", "Keep", "Valid extreme events; deleting them would understate tail risk (VaR/ES)."),
 ("DS2", "Volatility columns are decimals (e.g. 0.31), data dictionary says '% annualised' and uses different column names (vol_5d etc.)",
  "Treat *_ann columns as decimal annualised vol (verified vs rolling std*sqrt(252), rule DS2-10 = 0 violations)", "Recomputation matches the decimal scale exactly."),
 ("DS2", "All weekdays present incl. common US holidays (see sec. 9); no volume=0, no gaps beyond weekends",
  "Keep calendar as supplied; document as assumption that prices are a simulated/continuous weekday series", "Real exchange calendars have holiday closures; this affects nothing mechanically but limits claims of 'market realism'."),
 ("DS3", f"condition_rating=0 in {R['DS3-01']} rows (README scale 1-9)",
  "Keep original; add condition_rating_clipped=max(rating,1) for ordinal use", "0 is below the documented scale, but these rows are all structurally_deficient=1 and the lowest condition; 0.1% of data."),
 ("DS3", f"Inspection date earlier than year_built in {R['DS3-11']} rows (young bridges)",
  "Keep; flag; do not use last_inspection_date as predictor", "age_years is internally consistent with year_built; the date field is the suspect one."),
 ("DS3", f"structurally_deficient == (condition_rating<=4) exactly ({int(R['DS3-08'])} mismatches)",
  "Treat as derived label; never include both in a model of the other", "Deterministic function of rating -> target leakage."),
 ("DS3", f"remaining_thickness == original - corrosion_rate*age (only {R['DS3-10']} exception)",
  "Use thickness fraction OR corrosion*age, not independent evidence of both", "Derived variable; double counting inflates apparent degradation signal and creates collinearity."),
 ("DS3", f"age>design_life in {R['DS3-09']} rows ({100*R['DS3-09']/len(d3):.1f}%)", "Keep; engineer age/design_life ratio", "Not an error: legitimate exposure indicator."),
 ("DS3", "paris_C == 0 / paris_m == 0 for non-metals (concrete, timber, FRP)", "Keep as 'not applicable'; fatigue features only for metals + indicator", "Data dictionary: Paris constants N/A for concrete; zeros are not missing measurements."),
 ("DS4", "Trend features equal mqi - mqi.shift(k) (backward-looking, rule DS4-06)", "Safe as time-t features; build forward targets separately and shift", "Prevents look-ahead leakage when merging with DS2."),
 ("DS5", f"monthly_return irreconcilable with price path ({d5_mismatch}/{n5} rows mismatch, ~0 correlation, std 0.24 vs price-derived std {cand['simple return t/t-1'].std():.3f})",
  "Do NOT use supplied monthly_return; derive returns from price_usd_per_kg", "Documented README key column, but cannot be reproduced from supplied prices; using it would inject unexplained noise."),
 ("DS5", "Cheap elements have coarse price resolution (many unchanged months, e.g. Fe 0.1 $/kg)", "Keep; interpret returns for low-priced elements with caution", "Rounding artefact: apparent zero volatility is not real stability."),
 ("DS6", f"loss_ratio>1 in {R['DS6-02']} rows (repair > replacement)", "Keep raw; carry loss_ratio and loss_ratio_capped=min(lr,1) and report LGD under both", "Plausible for catastrophic events (cost overruns); cap choice is a modelling assumption that must be shown, not hidden."),
 ("DS6", f"was_predicted==1 with warning_lead_months==0 in {R['DS6-04']} rows", "Keep; exclude was_predicted/warning_lead_months/detected_by from failure-model features", "Post-event information; ambiguous semantics for 51 rows."),
 ("DS6", "Only failure events exist (no non-failures, no censoring)", "Cannot train a stand-alone classifier or survival model on DS6; design case-control/relative-risk approach with explicit assumptions (see architecture)", "A hazard needs exposure/denominator data; DS6 supplies only the numerator."),
 ("XLSX", "Workbook samples are separate entities from the CSVs (0 matching material ids; asset ids BR-001 vs BR-00001; DS4 snapshot values differ from CSV on 2024-01-15)",
  "Use workbook only as SPECIFICATION (stress-scenario parameters, 10-asset simulator portfolio, validation checks) - do not merge with CSVs", "No shared keys or values; merging would be fabricating a relationship."),
], columns=["dataset","PROBLEM","DECISION","REASON"])
decisions.to_csv(TAB/"cleaning_decisions.csv", index=False); display(decisions)

,dataset,PROBLEM,DECISION,REASON
0,DS1,material_id duplicated: 42 rows / 21 ids with conflicting formulas,Keep all rows; use positional row_id as the key; never join on material_id,Cannot tell which record is correct; dropping either would be arbitrary. Joi...
1,DS1,498 repeated formulas with large property spread (median K CV 0.50),Keep; use GroupKFold by formula in all CV,Same-formula rows in train and test would leak; rows are not interchangeable...
2,DS1,is_stable==1 with formation_energy>=0: 369 rows (README says stable => Ef<0),Keep; do not treat is_stable as a clean ground truth; report model skill aga...,Label contradicts the documented definition for 6.7% of rows; cannot repair ...
3,DS1,"Elastic identity violated: nu vs nu(K,G) >0.01 in 1582 rows (28.8%); E(G,nu)...",Keep; model K and G as primary targets and DERIVE nu and E from them so pred...,README demands E=2G(1+nu)=3K(1-2nu); supplied triplets are mutually inconsis...
4,DS1,"Values beyond documented 'typical' ranges: Ef<-5 in 126 rows, K>450 in 10, G...",Keep; flag; use robust/transform-aware modelling,"Soft ranges ('typically'); high-K/G (e.g. carbides, borides) are physically ..."
5,DS2,NaNs in rolling indicators (max 504 per column),Keep as NaN; no imputation; drop warm-up rows only inside feature/target win...,Section 7 shows NaNs exist only in the first k rows of each commodity (rolli...
6,DS2,|daily_return|>10% in 2 rows,Keep,Valid extreme events; deleting them would understate tail risk (VaR/ES).
7,DS2,"Volatility columns are decimals (e.g. 0.31), data dictionary says '% annuali...",Treat *_ann columns as decimal annualised vol (verified vs rolling std*sqrt(...,Recomputation matches the decimal scale exactly.
8,DS2,"All weekdays present incl. common US holidays (see sec. 9); no volume=0, no ...",Keep calendar as supplied; document as assumption that prices are a simulate...,Real exchange calendars have holiday closures; this affects nothing mechanic...
9,DS3,condition_rating=0 in 5 rows (README scale 1-9),"Keep original; add condition_rating_clipped=max(rating,1) for ordinal use","0 is below the documented scale, but these rows are all structurally_deficie..."


## 18 · Readable report: `outputs/data_quality_report.html`

In [23]:
def h(df, **kw): return df.to_html(index=False, border=0, na_rep="", float_format=lambda x: f"{x:,.4g}", **kw)
sections = [
 ("Scope", "<p>Automated audit of the 6 supplied CSV datasets and the xlsx workbook. All numbers are computed from the files at run time. Raw files were not modified (SHA-256 verified).</p>"),
 ("Dataset summary", h(summary)),
 ("Data-quality rules with violations", h(rules[rules.n_violations > 0][["rule_id","kind","description","source","n_violations","pct_violations"]])),
 ("Rules that passed (0 violations)", "<p>" + ", ".join(rules[rules.n_violations==0].rule_id) + "</p>"),
 ("Cleaning decisions (PROBLEM -> DECISION -> REASON)", h(decisions)),
 ("Relationships supported by the data", h(rel)),
 ("Candidate targets / leakage-risk columns", h(targets)),
 ("Outliers (3xIQR, flagged only)", h(out_all[out_all.n_extreme>0].sort_values("pct_extreme", ascending=False).head(30))),
]
A.render_html_report(ROOT/"outputs/data_quality_report.html", "MatRisk AI - Data Quality Report", sections)
A.render_html_report(REP/"data_quality_report.html", "MatRisk AI - Data Quality Report", sections)
print("written:", ROOT/"outputs/data_quality_report.html", "|", (ROOT/"outputs/data_quality_report.html").stat().st_size//1024, "KB")

written: /home/claude/matrisk/outputs/data_quality_report.html | 27 KB


## 19 · Integrity check: raw data untouched

In [24]:
RAW_HASH_END = {p.name: sha(p) for p in sorted(RAW.iterdir())}
assert RAW_HASH_START == RAW_HASH_END, "RAW FILES CHANGED"
pd.DataFrame({"file": list(RAW_HASH_END), "sha256": list(RAW_HASH_END.values())}).to_csv(TAB/"raw_file_hashes.csv", index=False)
print("Raw files unchanged. Hashes saved to outputs/tables/raw_file_hashes.csv")

Raw files unchanged. Hashes saved to outputs/tables/raw_file_hashes.csv


## Audit conclusions (all supported by outputs above)
1. **Joins that exist:** DS2↔DS4 on `(date, commodity)` is an exact 1:1 join (22,952 matched rows). DS3↔DS6 can only be linked on *categorical segments* (material, location, corrosion_environment) — there is **no asset-level key**.
2. **Chains that do NOT exist:** no key links DS1 (crystalline compounds) to DS3/DS6 (engineering material grades) or to any commodity; DS5 elements link to DS1 only through parsed formulas and cover just ~33% of DS1 materials fully; DS5 does not co-move with DS2.
3. **DS6 has failures only** — it cannot, alone, support a classifier, survival model or any absolute failure probability. This is the central modelling constraint for the risk engine.
4. **Physics constraints in the README are violated by the data itself** (elastic identity, `is_stable` vs formation energy) → the material model must enforce physics at prediction time instead of trusting labels.
5. **DS5 `monthly_return` is unusable;** DS3 `structurally_deficient` and `remaining_thickness_mm` are derived; DS6 post-event columns are leakage.
6. The statistical-structure preview (section 14) shows weak cross-property correlations in DS1 and near-zero cross-commodity correlation (mean 0.008), near-zero MQI→forward-return correlation (0.013), and ≈0 excess kurtosis in every commodity's returns (Gaussian-like tails, no fat-tail history). Model results must be judged against that ceiling, and tail risk cannot be learned from this history — it must come from the workbook's stress-scenario parameters.